In [3]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import GEOparse
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import gseapy

In [8]:
file_path = "../data/raw/GSE191328_series_matrix.txt.gz"

print(file_path)

../data/raw/GSE191328_series_matrix.txt.gz


In [18]:
# Download and parse the Crohn's disease dataset from GEO
import GEOparse

gse_cd = GEOparse.get_GEO(
    "GSE107865",
    destdir="../data/raw",
    how="brief",
    annotate_gpl=False
)

print("Total samples:", len(gse_cd.gsms))

01-Nov-2026 14:57:44 DEBUG utils - Directory ../data/raw already exists. Skipping.
01-Nov-2026 14:57:44 INFO GEOparse - Downloading http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?targ=gsm&acc=GSE107865&form=text&view=brief to ../data/raw\GSE107865.txt
01-Nov-2026 14:57:45 DEBUG downloader - Total size: 0
01-Nov-2026 14:57:45 DEBUG downloader - md5: None
110kB [00:00, 113kB/s]  
01-Nov-2026 14:57:46 DEBUG downloader - Moving C:\Users\Sofie\AppData\Local\Temp\tmpby67oht_ to c:\crohns-treatment-response-ml\data\raw\GSE107865.txt
01-Nov-2026 14:57:46 DEBUG downloader - Successfully downloaded http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?targ=gsm&acc=GSE107865&form=text&view=brief
01-Nov-2026 14:57:46 INFO GEOparse - Parsing ../data/raw\GSE107865.txt: 
01-Nov-2026 14:57:46 DEBUG GEOparse - SAMPLE: GSM2882184
01-Nov-2026 14:57:46 DEBUG GEOparse - SAMPLE: GSM2882185
01-Nov-2026 14:57:46 DEBUG GEOparse - SAMPLE: GSM2882186
01-Nov-2026 14:57:46 DEBUG GEOparse - SAMPLE: GSM2882187
01-Nov-2026 14

Total samples: 22


In [19]:
# Collect sample information from GEO
sample_data = []

for gsm_id, gsm in gse_cd.gsms.items():
    sample_data.append({
        "sample_id": gsm_id,
        "title": gsm.metadata.get("title", [""])[0],
        "source": gsm.metadata.get("source_name_ch1", [""])[0],
        "characteristics": "; ".join(
            gsm.metadata.get("characteristics_ch1", [])
        )
    })

# Create a sample metadata table
metadata_cd = pd.DataFrame(sample_data)

print("Metadata shape:", metadata_cd.shape)
metadata_cd.head()

Metadata shape: (22, 4)


,sample_id,title,source,characteristics
0,GSM2882184,HR-42-V1,peripheral blood mononuclear cells,disease state: Crohn's disease;treatment:none;patient:HR-42;time:V1;status:Responder;CRP:3.44;HBI:0;BMI:23.038752362949;status_time:W14; treatment: none; patient: HR-42; time: V1; status: Responder; crp: 3.44; hbi: 0; bmi: 23.038752362949; status_time: W14
1,GSM2882185,HR-44-V1,peripheral blood mononuclear cells,disease state: Crohn's disease;treatment:none;patient:HR-44;time:V1;status:Responder;CRP:19.6;HBI:2;BMI:26.6727632982777;status_time:W14; treatment: none; patient: HR-44; time: V1; status: Responder; crp: 19.6; hbi: 2; bmi: 26.6727632982777; status_time: W14
2,GSM2882186,HR-46-V1,peripheral blood mononuclear cells,disease state: Crohn's disease;treatment:none;patient:HR-46;time:V1;status:Responder;CRP:6.82;HBI:2;BMI:26.643598615917;status_time:W14; treatment: none; patient: HR-46; time: V1; status: Responder; crp: 6.82; hbi: 2; bmi: 26.643598615917; status_time: W14
3,GSM2882187,HR-47-V1,peripheral blood mononuclear cells,disease state: Crohn's disease;treatment:none;patient:HR-47;time:V1;status:Responder;CRP:125.7;HBI:6;BMI:21.875;status_time:W14; treatment: none; patient: HR-47; time: V1; status: Responder; crp: 125.7; hbi: 6; bmi: 21.875; status_time: W14
4,GSM2882188,HR-48-V1,peripheral blood mononuclear cells,disease state: Crohn's disease;treatment:none;patient:HR-48;time:V1;status:Responder;CRP:1.8;HBI:0;BMI:23.0400201994698;status_time:W14; treatment: none; patient: HR-48; time: V1; status: Responder; crp: 1.8; hbi: 0; bmi: 23.0400201994698; status_time: W14


In [20]:
# Display the full sample characteristics
for i in range(5):
    print(metadata_cd.loc[i, "sample_id"])
    print(metadata_cd.loc[i, "characteristics"])
    print("-" * 60)

GSM2882184
disease state: Crohn's disease;treatment:none;patient:HR-42;time:V1;status:Responder;CRP:3.44;HBI:0;BMI:23.038752362949;status_time:W14; treatment: none; patient: HR-42; time: V1; status: Responder; crp: 3.44; hbi: 0; bmi: 23.038752362949; status_time: W14
------------------------------------------------------------
GSM2882185
disease state: Crohn's disease;treatment:none;patient:HR-44;time:V1;status:Responder;CRP:19.6;HBI:2;BMI:26.6727632982777;status_time:W14; treatment: none; patient: HR-44; time: V1; status: Responder; crp: 19.6; hbi: 2; bmi: 26.6727632982777; status_time: W14
------------------------------------------------------------
GSM2882186
disease state: Crohn's disease;treatment:none;patient:HR-46;time:V1;status:Responder;CRP:6.82;HBI:2;BMI:26.643598615917;status_time:W14; treatment: none; patient: HR-46; time: V1; status: Responder; crp: 6.82; hbi: 2; bmi: 26.643598615917; status_time: W14
------------------------------------------------------------
GSM2882187


In [21]:
# Display treatment, response and timepoint information for all samples
for i in range(len(metadata_cd)):
    print(
        metadata_cd.loc[i, "sample_id"],
        metadata_cd.loc[i, "characteristics"]
    
    )
    print("-" *50)

GSM2882184 disease state: Crohn's disease;treatment:none;patient:HR-42;time:V1;status:Responder;CRP:3.44;HBI:0;BMI:23.038752362949;status_time:W14; treatment: none; patient: HR-42; time: V1; status: Responder; crp: 3.44; hbi: 0; bmi: 23.038752362949; status_time: W14
--------------------------------------------------
GSM2882185 disease state: Crohn's disease;treatment:none;patient:HR-44;time:V1;status:Responder;CRP:19.6;HBI:2;BMI:26.6727632982777;status_time:W14; treatment: none; patient: HR-44; time: V1; status: Responder; crp: 19.6; hbi: 2; bmi: 26.6727632982777; status_time: W14
--------------------------------------------------
GSM2882186 disease state: Crohn's disease;treatment:none;patient:HR-46;time:V1;status:Responder;CRP:6.82;HBI:2;BMI:26.643598615917;status_time:W14; treatment: none; patient: HR-46; time: V1; status: Responder; crp: 6.82; hbi: 2; bmi: 26.643598615917; status_time: W14
--------------------------------------------------
GSM2882187 disease state: Crohn's disease

In [23]:
# Extract response labels from sample characteristics
metadata_cd["response"] = metadata_cd["characteristics"].str.extract(
    r"status:\s*([^;]+)"
)

metadata_cd["timepoint"] = metadata_cd["characteristics"].str.extract(
    r"time:\s*([^;]+)"
)

print("Response counts:")
print(metadata_cd["response"].value_counts())

print("\nTimepoint counts:")
print(metadata_cd["timepoint"].value_counts())

Response counts:
response
Responder        17
Non-responder     5
Name: count, dtype: int64

Timepoint counts:
timepoint
V1    22
Name: count, dtype: int64


In [25]:
# Reload GEO with the full sample data tables
gse_cd_full = GEOparse.get_GEO(
    "GSE107865",
    destdir="../data/raw",
    how="full",
    annotate_gpl=False
)

# Check whether expression values are now available
sample = gse_cd_full.gsms["GSM2882184"]

print("Table shape:", sample.table.shape)
print(sample.table.head())

01-Nov-2026 15:38:02 DEBUG utils - Directory ../data/raw already exists. Skipping.
01-Nov-2026 15:38:02 INFO GEOparse - Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE107nnn/GSE107865/soft/GSE107865_family.soft.gz to ../data/raw\GSE107865_family.soft.gz
  0%|          | 0.00/13.0M [00:01<?, ?B/s]


OSError: Download failed due to 'Downloaded size do not match the expected size for ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE107nnn/GSE107865/soft/GSE107865_family.soft.gz'. ID could be incorrect or the data might not be public yet.

In [26]:
# Check the downloaded GEO Series Matrix file
import gzip

file_path = "../data/raw/GSE107865_series_matrix.txt.gz"

with gzip.open(file_path, "rt") as file:
    for i in range(12):
        print(file.readline().strip())

!Series_title	"PBMC gene expression from Crohn's disease patients before Infliximab therapy."
!Series_geo_accession	"GSE107865"
!Series_status	"Public on Feb 01 2018"
!Series_submission_date	"Dec 08 2017"
!Series_last_update_date	"Feb 01 2019"
!Series_pubmed_id	"29618496"
!Series_summary	"About 40% IBD patients treated with anti-TNF antibodies do not respond to therapy. Baseline biomarkers of response are therefore of interest. By combining computational deconvolution of gene expression and meta-analysis approaches we identified cellular biomarkers in tissue (validated in 2 cohorts by IHC of biopsies), and investigated associated gene biomarkers in blood. This dataset provides data from the validation cohort III (blood)."
!Series_overall_design	"Whole blood samples were collected from 22 patients diagnosed with Crohn's disease, just before starting anti-TNF therapy (Infliximab). All patients were naive to anti-TNF. Response to therapy was assessed after 14 weeks based on changes in CRP

In [27]:
#Find where the gene expression table begins
with gzip.open(file_path, "rt") as file:
    for line in file:
        if "!series_matrix_table_begin" in line.lower():
            print("Expression table found!")
            print(line.strip())
            break

Expression table found!
!series_matrix_table_begin


In [28]:
#Read the gene expression matrix from the GEO file
with gzip.open(file_path, "rt") as file:
    lines = file.readlines()

# Find the beginning and end of the expression table
start = lines.index("!series_matrix_table_begin\n") + 1
end = lines.index("!series_matrix_table_end\n")

#Load expression values into a DataFrame
from io import StringIO

expression_data = pd.read_csv(
    StringIO("".join(lines[start:end])),
    sep="\t"
)

print("Expression data shape:", expression_data.shape)
expression_data.head()

Expression data shape: (24351, 23)


,ID_REF,GSM2882184,GSM2882185,GSM2882186,GSM2882187,GSM2882188,GSM2882189,GSM2882190,GSM2882191,GSM2882192,...,GSM2882196,GSM2882197,GSM2882198,GSM2882199,GSM2882200,GSM2882201,GSM2882202,GSM2882203,GSM2882204,GSM2882205
0,AFFX-BkGr-GC03_st,2.005046,1.983293,1.960426,2.035343,1.931051,1.791423,1.784147,1.848812,2.040481,...,1.999921,2.251883,2.302799,2.179057,2.081086,1.929881,2.022703,1.971961,2.157274,1.916600
1,AFFX-BkGr-GC04_st,1.880923,1.974961,1.941266,1.915327,1.842159,1.739075,1.831695,1.810775,1.794120,...,1.863624,1.907798,2.112753,1.976534,1.854095,1.833135,1.839680,1.875990,1.920233,1.857772
2,AFFX-BkGr-GC05_st,1.915240,1.927470,1.966480,1.971324,1.862916,1.751935,1.843181,1.775349,1.771925,...,1.869214,1.908775,2.192303,1.968472,1.851979,1.784312,1.844144,1.921917,1.910708,1.852143
3,AFFX-BkGr-GC06_st,1.944509,2.037043,1.996847,2.010977,1.927292,1.855461,1.874566,1.851702,1.841577,...,1.946240,2.023223,2.182571,2.125019,1.915011,1.820662,1.891860,1.939478,1.986499,1.944962
4,AFFX-BkGr-GC07_st,1.984249,2.012549,2.015486,2.032749,1.975525,1.844585,1.913244,1.882891,1.867593,...,1.947533,1.980183,2.228078,2.053120,1.954136,1.879987,1.934378,1.950612,2.019969,1.953265


In [29]:
#Check sample names and probe identifiers

print("Total columns:", len(expression_data.columns))
print("\nColumn names:")
print(expression_data.columns.tolist())

print("\nFirst 10 probe IDs:")
print(expression_data["ID_REF"].head(10))

Total columns: 23

Column names:
['ID_REF', 'GSM2882184', 'GSM2882185', 'GSM2882186', 'GSM2882187', 'GSM2882188', 'GSM2882189', 'GSM2882190', 'GSM2882191', 'GSM2882192', 'GSM2882193', 'GSM2882194', 'GSM2882195', 'GSM2882196', 'GSM2882197', 'GSM2882198', 'GSM2882199', 'GSM2882200', 'GSM2882201', 'GSM2882202', 'GSM2882203', 'GSM2882204', 'GSM2882205']

First 10 probe IDs:
0    AFFX-BkGr-GC03_st
1    AFFX-BkGr-GC04_st
2    AFFX-BkGr-GC05_st
3    AFFX-BkGr-GC06_st
4    AFFX-BkGr-GC07_st
5    AFFX-BkGr-GC08_st
6    AFFX-BkGr-GC09_st
7    AFFX-BkGr-GC10_st
8    AFFX-BkGr-GC11_st
9    AFFX-BkGr-GC12_st
Name: ID_REF, dtype: str


In [31]:
# Check that expression samples match metadata samples
expression_samples = expression_data.columns[1:].tolist()
metadata_samples = metadata_cd["sample_id"].tolist()

print("Expression samples:", len(expression_samples))
print("Metadata samples:", len(metadata_samples))
print("All samples match:", set(expression_samples) == set(metadata_samples))

# Save expression data and sample metadata
expression_data.to_csv("../data/processed/GSE107865_expression.csv", index=False)
metadata_cd.to_csv("../data/processed/GSE107865_metadata.csv", index=False)

print("\nFiles saved successfully.")


Expression samples: 22
Metadata samples: 22
All samples match: True

Files saved successfully.
